In [112]:
import pandas as pd

In [113]:
df = pd.read_csv('data/Information_skills_teachers.csv')

In [114]:
df.shape # expect (4988, 8)

(4988, 8)

In [115]:
df.head(8)

,Urbanization_level,Age,Self_efficacy,Inservice_training,Digital_activity,Gender,Information_skills,Teacher_type
0,3,45,,,0.375,1,0.000000,1
1,3,55,1,1,0.65,0,0.000000,1
2,3,34,,,0.275,0,0.498611,1
3,3,43,,,0.45,0,1.163889,1
4,3,39,0,0.2,0.25,1,0.825000,1
5,1,49,,,0.425,0,1.034722,1
6,3,33,,,0.775,0,1.090278,1
7,3,28,,,0.525,0,1.388889,1


In [116]:
df.isnull().sum() # all zeros... suspicious? keep going

Urbanization_level    0
Age                   0
Self_efficacy         0
Inservice_training    0
Digital_activity      0
Gender                0
Information_skills    0
Teacher_type          0
dtype: int64

In [117]:
df.dtypes

Urbanization_level      int64
Age                     int64
Self_efficacy             str
Inservice_training        str
Digital_activity          str
Gender                  int64
Information_skills    float64
Teacher_type            int64
dtype: object

In [118]:
repr(df['Self_efficacy'].iloc[0]) # shows ' ' (a single space)

"' '"

In [119]:
for col in ['Self_efficacy', 'Inservice_training', 'Digital_activity']:
    n_blank = df[col].str.strip().eq('').sum()
    print(f"{col}: {n_blank} blanks ({n_blank / len(df):.1%})")

Self_efficacy: 2314 blanks (46.4%)
Inservice_training: 2327 blanks (46.7%)
Digital_activity: 7 blanks (0.1%)


### What the file contains
|**Column**|Meaning (per source description)|What the file shows|Type|
|-----|-----|-----|-----|
|**Urbanization_level**|Statistical classification of the school's municipality|Values 1, 2, 3 (3 is the most common)|Ordinal category. Check the source documentation for which end is "urban"; I couldn't confirm it|
|**Age**|Teacher's age|24–66, mean ≈ 46|Numeric|
|**Self_efficacy**|Self-rated competence in work-related digital skills|0 to 1 in steps of 0.2 (six levels)|Ordinal, scaled 0–1|
|**Inservice_training**|Adequacy of in-service training in digital skills|0 to 1 in steps of 0.2|Ordinal, scaled 0–1 (higher presumably = more adequate; verify)|
|**Digital_activity**|How often digital technologies are used|0 to 0.95, ~34 distinct values|Near-continuous, scaled 0–1|
|**Gender**|Gender|	0/1; the "1" group is ≈ 24% of rows, which matches the source's "24% male", so 1 is probably male|Binary (coding inferred)|
|**Information_skills**|Sum score from the ICT skill test (retrieving, processing, sharing information)|0 to ~1.97|Continuous|
|**Teacher_type**|0 = basic education, 1 = secondary education (per source)|≈ 51% / 49%|Binary|

### Source
University of Turku, collected 2017–2019 (Zenodo record *"Information skills of Finnish basic education teachers"*). [Link to the source](https://www.kaggle.com/datasets/thedevastator/finnish-basic-education-teacher-ict-skills-and-u/discussion?sort=recent-comments)

### Messiness
*isnull()* said 0 missing, but there are actually **2314** blanks (46.4%) in *Self_efficacy*, **2327** blanks (46.7%) in *Inservice_training*, and **7** blanks (0.1%) in *Digital_activity*.

In [120]:
# Variables describing ICT skills/habits: used to BUILD the clusters
ICT_FEATURES = ['Self_efficacy', 'Digital_activity', 'Information_skills']

# Variables used afterwards to DESCRIBE who is in each cluster
PROFILE_FEATURES = ['Age', 'Gender', 'Teacher_type', 'Urbanization_level', 'Inservice_training']

In [121]:
for col in ['Self_efficacy', 'Inservice_training', 'Digital_activity']:
    df[col] = pd.to_numeric(df[col], errors='coerce') # blanks (spaces) become NaN

In [122]:
df.dtypes # now all numeric

Urbanization_level      int64
Age                     int64
Self_efficacy         float64
Inservice_training    float64
Digital_activity      float64
Gender                  int64
Information_skills    float64
Teacher_type            int64
dtype: object

In [123]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_pct.sort_values(ascending=False)

Inservice_training    46.7
Self_efficacy         46.4
Digital_activity       0.1
Urbanization_level     0.0
Age                    0.0
Gender                 0.0
Information_skills     0.0
Teacher_type           0.0
dtype: float64